# Gemma 4 Developer Agent: Submission Notebook

Clean notebook meant to run **top to bottom with no manual steps**. It installs the offline wheels, copies my submission files from my private Kaggle dataset, optionally runs a short dry run, and packages `submission.zip`.

**Before running (Input panel and settings):**
1. Attach: the competition, the wheelhouse dataset, the Gemma 4 model, and **my private submission dataset** (the one containing `agent.yaml`).
2. Internet: off. Accelerator: GPU L4 x4.

Everything after the install cell is written so that a problem in the dry run is printed but never stops the packaging cell. The dry run is **off by default** (`RUN_SMOKE_TEST = False`) because the model server can take 20+ minutes to start and is not needed to build `submission.zip`.

In [ ]:
# ---- Submission notebook settings ----
RUN_SMOKE_TEST = False   # True: start the model server and run a short dry run. False: skip both and just package (fastest, lowest risk).
NUM_SMOKE_TASKS = 2      # how many tasks the dry run evaluates
SERVER_STARTUP_MINUTES = 50  # a cold model load has taken over 20 minutes, so allow plenty of time

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and My Submission

In [ ]:
# --- Guard: the install cell must have run first ---
try:
    import swegemma
except ModuleNotFoundError:
    raise RuntimeError(
        "swegemma is not installed. Run the 'Environment Configuration and Package "
        "Installation' cell first, wait for 'Wheelhouse installation complete.', "
        "then rerun this cell."
    )

from swegemma.models import load_tasks
import traceback
import zipfile
import shutil
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Directories used by the code-graph tools
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# --- Locate MY submission (the attached dataset that contains agent.yaml, any folder depth) ---
# To override auto-detection, set it by hand, e.g.:
# MY_SUBMISSION_SRC = Path('/kaggle/input/datasets/<owner>/<slug>/<folder>')
MY_SUBMISSION_SRC = None

SKIP_PARTS = {'competitions', 'gemma-4-developer-agent', 'models'}  # never pick the competition's own sample

if MY_SUBMISSION_SRC is None:
    INPUT_ROOT = Path('/kaggle/input')
    hits = []
    for ds_dir in INPUT_ROOT.glob('datasets/*/*'):
        hits += list(ds_dir.rglob('agent.yaml'))
    for ds_dir in INPUT_ROOT.glob('*'):  # older-style mounts
        if ds_dir.name not in SKIP_PARTS and ds_dir.name != 'datasets':
            hits += list(ds_dir.rglob('agent.yaml'))
    hits = [p for p in hits if not (set(p.parts) & SKIP_PARTS)]
    hits.sort(key=lambda p: len(p.parts))  # shallowest first

    if hits:
        if len(hits) > 1:
            print('Multiple agent.yaml files found, using the shallowest:')
            for h in hits:
                print('  ', h)
        MY_SUBMISSION_SRC = hits[0].parent
    else:
        # Fallback: the dataset may have kept my files as a .zip
        for ds_dir in INPUT_ROOT.glob('datasets/*/*'):
            for z in ds_dir.rglob('*.zip'):
                with zipfile.ZipFile(z) as zf:
                    if any(n.endswith('agent.yaml') for n in zf.namelist()):
                        extract_dir = Path('/tmp/my_submission_src')
                        shutil.rmtree(extract_dir, ignore_errors=True)
                        zf.extractall(extract_dir)
                        found = sorted(extract_dir.rglob('agent.yaml'), key=lambda p: len(p.parts))
                        MY_SUBMISSION_SRC = found[0].parent
                        break
            if MY_SUBMISSION_SRC is not None:
                break

if MY_SUBMISSION_SRC is None:
    listing = sorted(str(p) for p in Path('/kaggle/input').glob('*/*')) + \
              sorted(str(p) for p in Path('/kaggle/input').glob('datasets/*/*'))
    raise FileNotFoundError(
        'Could not find agent.yaml in any attached dataset. '
        'Attach my private submission dataset in the Input panel. Found:\n' + '\n'.join(listing)
    )

print(f'Using submission source: {MY_SUBMISSION_SRC}')

# Copy MY submission into a writable directory (this is what gets packaged)
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(MY_SUBMISSION_SRC, AGENT_DIR)
(AGENT_DIR / 'dataset-metadata.json').unlink(missing_ok=True)  # keep stray metadata out of the zip

# Sanity checks on the submission itself
assert (AGENT_DIR / 'agent.yaml').exists(), 'agent.yaml is not at the top level of the submission'
assert (AGENT_DIR / 'configs' / 'sampling.yaml').exists(), 'configs/sampling.yaml is missing'

# Load tasks (only used by the dry run; a problem here must not stop packaging)
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
try:
    tasks = load_tasks(TASKS_PATH)
except Exception:
    tasks = []
    print('WARNING: could not load tasks, so the dry run will be skipped.')
    traceback.print_exc()

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks')
for t in tasks[:5]:
    print(f"  - {getattr(t, 'instance_id', '?')} ({getattr(t, 'repo', '?')})")

## 3. Start vLLM Server (dry run only)

In [ ]:
import traceback

SERVER_OK = False
server_instance = None

if not RUN_SMOKE_TEST:
    print('RUN_SMOKE_TEST is False: skipping the model server.')
else:
    try:
        import litellm
        import torch
        import yaml
        from adk_submission import VllmConfig, VllmServer, discover_adapters
        from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

        litellm.drop_params = True

        TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
        MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
        INFERENCE_API_KEY = 'EMPTY'

        # Validate the single declared base model. Newer swegemma versions no longer ship
        # swegemma.models.discovery, so fall back to reading the yaml files directly.
        try:
            from swegemma.models.discovery import validate_single_declared_model
            declared_model = validate_single_declared_model(AGENT_DIR)
        except ImportError:
            class TolerantLoader(yaml.SafeLoader):
                pass

            def _any_custom_tag(loader, tag_suffix, node):
                if isinstance(node, yaml.ScalarNode):
                    return loader.construct_scalar(node)
                if isinstance(node, yaml.SequenceNode):
                    return loader.construct_sequence(node)
                return loader.construct_mapping(node)

            TolerantLoader.add_multi_constructor('!', _any_custom_tag)

            declared_models = set()
            yaml_files = [AGENT_DIR / 'agent.yaml'] + sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))
            for yf in yaml_files:
                cfg = yaml.load(yf.read_text(), Loader=TolerantLoader) or {}
                if isinstance(cfg, dict) and cfg.get('model'):
                    declared_models.add(str(cfg['model']))
            assert len(declared_models) == 1, f'Expected exactly one declared base model, found: {declared_models}'
            declared_model = declared_models.pop()
            assert declared_model == TARGET_MODEL_NAME, f'Unexpected model: {declared_model}'
            print('Declared base model:', declared_model)

        # Give vLLM and the wrapper enough time for a slow, cold model load
        import os
        os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = str(SERVER_STARTUP_MINUTES * 60)

        # Discover any PEFT LoRA adapters in the submission directory
        adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

        gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
        tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

        vllm_cfg = VllmConfig(
            model=str(MODEL_PATH),
            port=8000,
            host='127.0.0.1',
            tool_call_parser='gemma4',
            reasoning_parser='gemma4',
            max_model_len=32768,
            dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
            gpu_memory_utilization=0.90,
            enable_auto_tool_choice=True,
            enable_lora=True,
            max_loras=8,
            max_lora_rank=128,
            tensor_parallel_size=tp_size,
            startup_timeout=60 * SERVER_STARTUP_MINUTES,
        )
        server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
        server_instance.start()
        print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

        # Register model and LoRA adapter aliases for Google ADK
        models = server_instance.create_model_registry(
            aliases=list(dict.fromkeys([declared_model, TARGET_MODEL_NAME])),
            model_prefix='openai/',
            api_key=INFERENCE_API_KEY,
        )
        SERVER_OK = True
    except Exception:
        print('WARNING: the model server did not start. Continuing so the submission still gets packaged.')
        traceback.print_exc()

## 4. Dry Run: Phase 1 Inference and Phase 2 Verification (a few tasks)

In [ ]:
import traceback

if not (RUN_SMOKE_TEST and SERVER_OK and len(tasks) > 0):
    print('Skipping the dry run (smoke test off, model server not running, or no tasks loaded).')
else:
    try:
        import asyncio
        import concurrent.futures
        import pandas as pd
        import yaml
        from google.adk.agents.context_cache_config import ContextCacheConfig
        from google.adk.apps._configs import EventsCompactionConfig
        from swegemma.config import EvalConfig, build_submission_limits
        from swegemma.evaluate import Evaluator


        def run_sync(coro_or_fn, *args, **kwargs):
            """Run an async coroutine synchronously inside a notebook event loop."""
            fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
            try:
                loop = asyncio.get_running_loop()
            except RuntimeError:
                loop = None
            if loop is not None and loop.is_running():
                with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
                    return pool.submit(lambda: asyncio.run(fn())).result()
            return asyncio.run(fn())


        # Read evaluation settings from the submission's eval_config.yaml
        eval_config_file = AGENT_DIR / 'eval_config.yaml'
        raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
        eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

        timeout_seconds = int(eval_section.get('timeout_seconds', 300))
        max_tool_calls = int(eval_section.get('max_tool_calls', 100))
        max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
        turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
        max_turns = int(turns_raw) if turns_raw is not None else None
        print('eval_config.yaml ->', dict(timeout_seconds=timeout_seconds, max_tool_calls=max_tool_calls,
                                          max_time_minutes=max_time_minutes, max_turns=max_turns))

        # Select a small subset of tasks for the dry run
        SAMPLE_TASKS = tasks[:NUM_SMOKE_TASKS]
        limits, gen_constraints = build_submission_limits()

        # Phase 1 (agent patch generation) and Phase 2 (verification) with the subprocess sandbox
        eval_config = EvalConfig(
            tasks_path=TASKS_PATH,
            snapshots_dir=DATA_DIR / 'snapshots',
            results_dir=WORKING_DIR / 'results',
            submission_dir=AGENT_DIR,
            models=models,
            sandbox='subprocess',
            timeout_seconds=timeout_seconds,
            max_time_minutes=max_time_minutes,
            max_tool_calls=max_tool_calls,
            max_turns=max_turns,
            limits=limits,
            generation_constraints=gen_constraints,
            adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
            events_compaction_config=EventsCompactionConfig(
                compaction_interval=15,
                overlap_size=2,
                token_threshold=14336,
                event_retention_size=5,
            ),
            graph_dir=GRAPH_DIR,
            embeddings_dir=EMBEDDINGS_DIR,
            wheels_dir=DATA_DIR / 'wheels',
            verbose=False,
        )

        evaluator = Evaluator(eval_config)
        predictions = []

        for idx, task in enumerate(SAMPLE_TASKS, start=1):
            print(f"[{idx}/{len(SAMPLE_TASKS)}] Evaluating {getattr(task, 'instance_id', '?')}...")
            result = run_sync(
                evaluator.evaluate_task,
                task=task,
                task_index=idx,
                total_tasks=len(SAMPLE_TASKS),
            )
            predictions.append({'id': task.instance_id, 'prediction': result.agent_patch or ''})
            print(
                f'  -> resolved={result.resolved}, '
                f'exit_code={result.test_exit_code}, '
                f'patch_chars={len(result.agent_patch or "")}, '
                f'tool_calls={result.tool_calls}, '
                f'duration={result.duration_seconds:.1f}s'
            )

        submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
        display(submission_df)
    except Exception:
        print('WARNING: the dry run failed. Continuing so the submission still gets packaged.')
        traceback.print_exc()

## 5. Package Submission Archive

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')
with zipfile.ZipFile(zip_path, 'r') as zf:
    print('Files in submission.zip:')
    for n in zf.namelist():
        print('  ', n)
